In [1]:
import math
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

In [2]:
# Define state schema for quadratic equation solver
class QuadraticState(TypedDict):
    a: float
    b: float
    c: float
    discriminant: float
    root_type: str
    roots: List[str]
    result: str

In [3]:
# Discriminant calculation node
def calculate_discriminant(state: QuadraticState) -> dict:
    a, b, c = state["a"], state["b"], state["c"]
    d = b**2 - 4 * a * c
    return {"discriminant": d}

# Branch Node 1: Real and distinct roots
def real_distinct_roots(state: QuadraticState) -> dict:
    a, b = state["a"], state["b"]
    d = state["discriminant"]
    r1 = (-b + math.sqrt(d)) / (2 * a)
    r2 = (-b - math.sqrt(d)) / (2 * a)
    roots = [f"{r1:.4f}", f"{r2:.4f}"]
    result = f"Two real and distinct roots: x1 = {r1:.4f}, x2 = {r2:.4f}"
    return {"root_type": "Real & Distinct", "roots": roots, "result": result}

# Branch Node 2: Real and equal roots
def real_equal_roots(state: QuadraticState) -> dict:
    a, b = state["a"], state["b"]
    r = -b / (2 * a)
    roots = [f"{r:.4f}"]
    result = f"One real repeated root: x = {r:.4f}"
    return {"root_type": "Real & Equal", "roots": roots, "result": result}

# Branch Node 3: Complex conjugate roots
def complex_roots(state: QuadraticState) -> dict:
    a, b = state["a"], state["b"]
    d = state["discriminant"]
    real_part = -b / (2 * a)
    imag_part = math.sqrt(-d) / (2 * a)
    r1 = f"{real_part:.4f} + {imag_part:.4f}i"
    r2 = f"{real_part:.4f} - {imag_part:.4f}i"
    roots = [r1, r2]
    result = f"Two complex conjugate roots: x1 = {r1}, x2 = {r2}"
    return {"root_type": "Complex", "roots": roots, "result": result}

In [4]:
# Router function for conditional edge
def check_discriminant(state: QuadraticState) -> str:
    d = state["discriminant"]
    if d > 0:
        return "real_distinct"
    elif d == 0:
        return "real_equal"
    else:
        return "complex"

In [5]:
# Initialize StateGraph
graph = StateGraph(QuadraticState)

# Add nodes
graph.add_node("calculate_discriminant", calculate_discriminant)
graph.add_node("real_distinct_roots", real_distinct_roots)
graph.add_node("real_equal_roots", real_equal_roots)
graph.add_node("complex_roots", complex_roots)

# Add standard edge from START to discriminant calculation
graph.add_edge(START, "calculate_discriminant")

# Add conditional routing edges based on discriminant
graph.add_conditional_edges(
    "calculate_discriminant",
    check_discriminant,
    {
        "real_distinct": "real_distinct_roots",
        "real_equal": "real_equal_roots",
        "complex": "complex_roots"
    }
)

# Connect output branch nodes to END
graph.add_edge("real_distinct_roots", END)
graph.add_edge("real_equal_roots", END)
graph.add_edge("complex_roots", END)

# Compile the graph into an executable workflow
workflow = graph.compile()

In [6]:
# Test Case 1: Real & Distinct Roots (D > 0)
# Equation: x^2 - 5x + 6 = 0
case1 = {"a": 1, "b": -5, "c": 6}
res1 = workflow.invoke(case1)
print("=== Case 1: Real & Distinct Roots ===")
print("Discriminant:", res1["discriminant"])
print("Result:", res1["result"])

=== Case 1: Real & Distinct Roots ===
Discriminant: 1
Result: Two real and distinct roots: x1 = 3.0000, x2 = 2.0000


In [7]:
# Test Case 2: Real & Equal Roots (D = 0)
# Equation: x^2 - 4x + 4 = 0
case2 = {"a": 1, "b": -4, "c": 4}
res2 = workflow.invoke(case2)
print("=== Case 2: Real & Equal Roots ===")
print("Discriminant:", res2["discriminant"])
print("Result:", res2["result"])

=== Case 2: Real & Equal Roots ===
Discriminant: 0
Result: One real repeated root: x = 2.0000


In [8]:
# Test Case 3: Complex Conjugate Roots (D < 0)
# Equation: x^2 + 2x + 5 = 0
case3 = {"a": 1, "b": 2, "c": 5}
res3 = workflow.invoke(case3)
print("=== Case 3: Complex Conjugate Roots ===")
print("Discriminant:", res3["discriminant"])
print("Result:", res3["result"])

=== Case 3: Complex Conjugate Roots ===
Discriminant: -16
Result: Two complex conjugate roots: x1 = -1.0000 + 2.0000i, x2 = -1.0000 - 2.0000i
